# GH-LR Simulator – Milestone 2
## Zintegrowany model logistyki humanitarnej i bezpieczeństwa żywnościowego

**Nowe warstwy względem M1:**
- 🏗️ Kolejki rozładunkowe w hubach (dok jako zasób SimPy)
- 📊 Model popytu: zapasy, kalorie/osobę, wskaźnik IPC
- 🎲 Stochastyczność: incydenty drogowe, zamknięcia przejść, awarie pojazdów
- 📈 Analityka KPI: mapa zatorów, timeline IPC, rozkłady kolejek
- 🔄 Multi-seed: powtarzalne eksperymenty ze statystykami


In [1]:
# Krok 1: Instalacja zależności (uruchom raz)
!pip install -q osmnx>=2.0 geopandas pyogrio networkx simpy pandas openpyxl matplotlib numpy


[notice] A new release of pip is available: 25.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Krok 2: Upload plików
# Wgraj do Colab:
#   - wszystkie *.py z src/ (w tym nowe: demand.py, stochastic.py, analytics.py, sim_m2.py, run_m2.py)
#   - Commodities-Received.xlsx, Commodities-Dispatching.xlsx do data/raw/
#   - unosat_gaza.zip do data/raw/
#   - gaza_drive.graphml do data/processed/ (jeśli masz; inaczej --build-graph)

import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('src', exist_ok=True)
print('Katalogi gotowe.')

Katalogi gotowe.


In [3]:
# Krok 3: Przygotowanie danych z Exceli
import sys
sys.path.insert(0, 'src')
import data

trucks, shares = data.prepare('2024-03', force=False)
print(f'Ciężarówek: {len(trucks)}')
print(shares.round(3))

Ciężarówek: 4993
              Hub_Rafah  Hub_Khan_Younis  Hub_Deir_al_Balah  Hub_Gaza_City  \
Kerem Shalom      0.883            0.042              0.042          0.028   
Rafah             0.883            0.042              0.042          0.028   
Kissufim          0.883            0.042              0.042          0.028   
Gate 96           0.883            0.042              0.042          0.028   
JLOTS             0.883            0.042              0.042          0.028   
Erez              0.883            0.042              0.042          0.028   
Western Erez      0.883            0.042              0.042          0.028   

              Hub_North_Jabalia  
Kerem Shalom              0.005  
Rafah                     0.005  
Kissufim                  0.005  
Gate 96                   0.005  
JLOTS                     0.005  
Erez                      0.005  
Western Erez              0.005  


In [ ]:
# Krok 4: Symulacja – Scenariusz A (Baseline)
# Odpowiednik: python src/run_m2.py --month 2024-03 --scenario A

import sys
sys.path.insert(0, 'src')
import numpy as np
import pandas as pd
import config
import damage, data, events as ev, network, router as router_mod
import sim_m2, demand as demand_mod, analytics, stochastic

MONTH = '2024-03'
SEED = 42
rng = np.random.default_rng(SEED)

trucks, shares = data.prepare(MONTH)
month_start = pd.Timestamp(f'{MONTH}-01', tz='UTC')
next_month = month_start + pd.offsets.MonthBegin(1)
month_days = (next_month - month_start).days

G = network.load_graph()
crossing_nodes = network.snap_points(G, config.CROSSINGS)
hub_nodes = network.snap_points(G, config.HUBS)

damage.set_base_weights(G)
gdf = damage.load_unosat()
dm_road = damage.DamageModel(G, damage.filter_as_of(gdf, month_start.strftime('%Y-%m-%d')))
wave_events = ev.unosat_wave_events(dm_road, gdf, month_start, next_month)

router = router_mod.FieldRouter(G, hub_nodes, next(iter(crossing_nodes.values())))
routes = network.compute_routes(G, crossing_nodes, hub_nodes, router)

# Zdarzenia stochastyczne
stochastic.CLOSED_CROSSINGS.clear()
busy = ev.busiest_edges(routes, {(c, h): trucks['crossing'].value_counts().get(c, 0)
                                  for c in crossing_nodes for h in hub_nodes}, k=len(G.edges()))
road_incidents = stochastic.generate_road_incidents(G, month_days, rng, busy_edges=busy)
crossing_closures = stochastic.generate_crossing_closures(crossing_nodes, month_days, rng)
all_events = list(wave_events) + road_incidents + crossing_closures
print(f'Zdarzenia łącznie: {len(all_events)}')

dm_demand = demand_mod.DemandModel()

log, edge_usage = sim_m2.run_simulation(
    trucks=trucks, G=G, router=router,
    crossing_nodes=crossing_nodes, hub_nodes=hub_nodes, hub_shares=shares,
    demand_model=dm_demand, events=all_events, damage_model=dm_road,
    month=MONTH, rng=rng,
)

print(f'\nSymulacja zakończona: {len(log)} ciężarówek, '
      f'{log["delivered"].sum()} dowiezionych ({100*log["delivered"].mean():.1f}%)')

UNOSAT: warstwa Damage_Sites_GazaStrip_20250404, 174526 punktów
UNOSAT: pominięto 40 punktów z klasą 6 ('No Visible Damage')
Zniszczenia na 2024-03-01: 127553 z 174486 punktów
Zniszczenia nałożone: 35376 z 36448 krawędzi przechodzi przez oczka ze zniszczeniami
Zdarzenia łącznie: 4633


In [ ]:
# Krok 5: Analityka KPI i wykresy
import os
OUTDIR = f'output/{MONTH}/scenario_A'
os.makedirs(OUTDIR, exist_ok=True)

from pathlib import Path
reporter = analytics.Reporter(Path(OUTDIR), MONTH)
reporter.run(
    truck_log=log,
    demand_model=dm_demand,
    hub_edge_usage=edge_usage,
    G=G,
)

# Wyświetl wykresy inline
from IPython.display import Image, display
for fname in ['kpi_queue_dist.png', 'kpi_ipc_timeline.png',
              'kpi_supply_coverage.png', 'kpi_congestion_map.png',
              'kpi_delivery_timeline.png']:
    path = f'{OUTDIR}/{fname}'
    if os.path.exists(path):
        print(f'\n--- {fname} ---')
        display(Image(path))

In [ ]:
# Krok 6: Tabela popytu i IPC
summary = dm_demand.summary()
print('Podsumowanie pokrycia popytu:')
display(summary)

history = dm_demand.report()
print('\nHistoria dobowa (pierwsze 15 wierszy):')
display(history.head(15))

In [ ]:
# Krok 7: Scenariusz B – Skrócenie czasu odprawy o 50%
# Hipoteza 1: zwiększenie przepustowości granicy zmniejsza deficyt żywnościowy

config.SERVICE_MEAN_MIN = 12.5  # 50% z 25 min

rng_b = np.random.default_rng(SEED)
stochastic.CLOSED_CROSSINGS.clear()
dm_demand_b = demand_mod.DemandModel()

# Przebuduj router (parametry trasy bez zmian, ale nowe losowania service_min)
log_b, edge_usage_b = sim_m2.run_simulation(
    trucks=trucks, G=G, router=router,
    crossing_nodes=crossing_nodes, hub_nodes=hub_nodes, hub_shares=shares,
    demand_model=dm_demand_b, events=list(wave_events),  # bez stochastic dla czystego porównania
    damage_model=dm_road, month=MONTH, rng=rng_b,
)

config.SERVICE_MEAN_MIN = 25.0  # przywróć domyślne

print('Scenariusz A vs B:')
comparison = pd.DataFrame({
    'Scenariusz A (baseline)': [
        f"{log['delivered'].mean():.1%}",
        f"{log['queue_min'].mean():.1f} min",
        f"{dm_demand.summary()['food_coverage_pct'].mean():.1f}%",
    ],
    'Scenariusz B (odprawa -50%)': [
        f"{log_b['delivered'].mean():.1%}",
        f"{log_b['queue_min'].mean():.1f} min",
        f"{dm_demand_b.summary()['food_coverage_pct'].mean():.1f}%",
    ],
}, index=['Dowiezionych %', 'Śr. czas kolejki', 'Pokrycie popytu na żywność'])
display(comparison)

In [ ]:
# Krok 8: Multi-seed – weryfikacja stabilności wyników
N_SEEDS = 5  # zwiększ do 20+ dla publikacji

results = []
for i in range(N_SEEDS):
    rng_i = np.random.default_rng(SEED + i)
    stochastic.CLOSED_CROSSINGS.clear()
    dm_i = demand_mod.DemandModel()
    road_inc_i = stochastic.generate_road_incidents(G, month_days, rng_i, busy_edges=busy)
    cross_cl_i = stochastic.generate_crossing_closures(crossing_nodes, month_days, rng_i)
    log_i, _ = sim_m2.run_simulation(
        trucks=trucks, G=G, router=router,
        crossing_nodes=crossing_nodes, hub_nodes=hub_nodes, hub_shares=shares,
        demand_model=dm_i, events=list(wave_events) + road_inc_i + cross_cl_i,
        damage_model=dm_road, month=MONTH, rng=rng_i,
    )
    results.append({
        'seed': SEED + i,
        'delivered_pct': round(100 * log_i['delivered'].mean(), 1),
        'queue_mean_min': round(log_i['queue_min'].mean(), 1),
        'travel_mean_min': round(log_i['travel_min'].mean(), 1),
        'food_coverage_pct': round(dm_i.summary()['food_coverage_pct'].mean(), 1),
        'mean_ipc': round(dm_i.report().groupby('day')['ipc_phase'].mean().mean(), 2),
    })
    print(f'Seed {SEED+i}: {results[-1]}')

ms_df = pd.DataFrame(results)
print('\nStatystyki multi-seed:')
display(ms_df.describe().round(2))